# AIkenGPT MMLU-style Evaluation — Choice-text likelihood scoring

このNotebookはOpenAI評価と同じ `MMLUEvaluator` を使います。
通常は次の **User settings** だけを変更し、GPU runtimeでRun allしてください。

処理順: Settings → Environment setup → Model loading → Dataset / evaluator setup
→ Preflight / manifest → Evaluation → Results


## 1. Settings

### User settings — 通常はこのセルだけを変更

In [8]:
from pathlib import Path

PROJECT_DIR = Path("/content/AIkenSGTv1_New")
MMLU_ROOT = Path("/content/mmlu-reference")
DATA_DIR = MMLU_ROOT / "data"

# Evaluation repository
if not (PROJECT_DIR / "mmlu_eval").is_dir():
    !rm -rf /content/AIkenSGTv1_New
    !git clone https://github.com/HayatoHongo/AIkenSGTv1.git /content/AIkenSGTv1_New
    !cd /content/AIkenSGTv1_New && git switch tayama

# MMLU reference repository
if not MMLU_ROOT.exists():
    !git clone https://github.com/hendrycks/test.git /content/mmlu-reference

# MMLU data (not included in the GitHub repository)
if not (DATA_DIR / "dev").is_dir() or not (DATA_DIR / "test").is_dir():
    !pip install -q datasets pandas
    !cd /content/mmlu-reference && python /content/AIkenSGTv1_New/download_mmlu_hf.py
OUTPUT_DIR = Path("/content/drive/MyDrive/aikengpt_results")

SUBJECT = None          # 例: "abstract_algebra"。Noneは全subject
NTRAIN = 0              # 0ならzero-shot
SAMPLE_FRAC = 0.10
SEED = 42
LIMIT = 10               # 0ならsampled questionsをすべて評価

# Noneならこのrun用に生成。同じ問題集合を再利用するときは既存manifestを指定。
MANIFEST_PATH = None

# Choice-text score:
#   "sum"  = candidate token log probabilitiesの合計。長い候補ほど不利になりやすい。
#   "mean" = token数で平均したlength-normalized score。
TEXT_REDUCTION = "mean"


Cloning into '/content/mmlu-reference'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 291 (delta 40), reused 26 (delta 26), pack-reused 205 (from 1)
Receiving objects: 100% (291/291), 2.30 MiB | 23.60 MiB/s, done.
Resolving deltas: 100% (61/61), done.
README.md: 100% 53.2k/53.2k [00:00<00:00, 99.7MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 198MB/s]

all/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/3.50M [00:00<?, ?B/s]s]
all/test-00000-of-00001.parquet: downloading bytes: 100% 3.49M/3.49M [00:01<00:00, 2.49MB/s,  339kB/s  ]
all/test-00000-of-00001.parquet: reconstructing file: 100% 3.50M/3.50M [00:01<00:00, 2.50MB/s,  340kB/s  ]

all/validation-00000-of-00001.parquet: downloading bytes:   0% 0.00/408k [00:00<?, ?B/s]s]
all/validation-00000-of-00001.parquet: downloading bytes: 100% 407k/407k [00:01<00:00, 313kB/s, 39.6kB/s  ]
all/validation-00000-of-00001.parq

### Project settings — 通常の問題セット変更では編集不要

In [10]:
# Repository / fixed AIkenGPT settings
MODEL_REPO = "HayatoHongo/AIkenSGTv1"
MODEL_REVISION = None   # 再現性を厳密にする場合はimmutable revisionを指定
MODEL_PATH = None       # local .safetensorsを使う場合だけPathを指定
MODEL_ID = "HayatoHongo/AIkenSGTv1:model_clean.safetensors"
TOKENIZER = "gpt2"

DEVICE = "cuda"
DTYPE = "float32"
BATCH_SIZE = 1
MAX_CONTEXT_LENGTH = 2048
CONTEXT_POLICY = "reduce"  # 共通2048-token budget内まで両modelで同じようにshot数を減らす
CONTEXT_TOKENIZER = "gpt2"
PERMUTATION_COUNT = 4
PERMUTATION_SEED = 0       # cyclic orderは固定。記録用で乱数には未使用

SCORING_METHOD = "choice_text"
OUTPUT_NAME = f"aikengpt_{NTRAIN}shot_text_{TEXT_REDUCTION}.csv"


## 2. Environment setup

In [11]:
%pip install -q numpy pandas tiktoken safetensors huggingface_hub


In [12]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "mmlu_eval").is_dir(), (
    "PROJECT_DIR must point to the repository containing mmlu_eval/"
)
assert DATA_DIR.is_dir(), "DATA_DIR must contain dev/ and test/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Model loading

In [13]:
import torch
import tiktoken
from huggingface_hub import hf_hub_download
from mmlu_eval.backends.aikengpt_backend import (
    AIkenGPTBackend,
    file_sha256,
    load_custom_checkpoint,
)

assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a Colab GPU runtime"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if MODEL_PATH is None:
    MODEL_PATH = Path(hf_hub_download(
        repo_id=MODEL_REPO,
        filename="model_clean.safetensors",
        revision=MODEL_REVISION,
    ))
else:
    MODEL_PATH = Path(MODEL_PATH)

checkpoint_hash = file_sha256(MODEL_PATH)
tokenizer = tiktoken.get_encoding(TOKENIZER)
model = load_custom_checkpoint(
    MODEL_PATH,
    device=DEVICE,
    dtype=DTYPE,
    max_context_length=MAX_CONTEXT_LENGTH,
)
print("Checkpoint SHA256:", checkpoint_hash)


GPU: NVIDIA L4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_clean.safetensors: reconstructing file:   0%|          |  0.00B / 10.5GB            

model_clean.safetensors: downloading bytes:           |  0.00B            

Checkpoint SHA256: c6f8693f1446f435fb34ba8f621516221fc6cddd0c82326c7a00e37633bfd473


## 4. Dataset / evaluator setup

In [14]:
from mmlu_eval import EvalConfig, MMLUEvaluator

eval_config = EvalConfig(
    ntrain=NTRAIN,
    sample_frac=SAMPLE_FRAC,
    seed=SEED,
    limit=LIMIT,
    subject=SUBJECT,
    permutation_count=PERMUTATION_COUNT,
    permutation_seed=PERMUTATION_SEED,
    context_policy=CONTEXT_POLICY,
    context_tokenizer=CONTEXT_TOKENIZER,
    max_context_length=MAX_CONTEXT_LENGTH,
)
evaluator = MMLUEvaluator(DATA_DIR, eval_config)


## 5. Preflight / manifest

In [15]:
from mmlu_eval.core import atomic_csv

# Existing manifest is authoritative. Otherwise create one once and reuse it below.
manifest = evaluator.manifest(MANIFEST_PATH)
preflight_manifest_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.manifest.csv")
preflight_prompts_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.prompts.jsonl")
atomic_csv(preflight_manifest_path, manifest)
evaluator.export_debug(manifest, preflight_prompts_path)
active_manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH is not None else preflight_manifest_path

first = manifest.iloc[0]
first_case = evaluator.cases(first.subject, int(first.test_index))[0]
print(f"Questions: {len(manifest)} / prompts: {len(manifest) * 4}")
print("Manifest:", active_manifest_path)
print("\nFirst prompt:\n")
print(first_case.prompt)


Questions: 10 / prompts: 40
Manifest: /content/drive/MyDrive/aikengpt_results/aikengpt_0shot_text_mean.preflight.manifest.csv

First prompt:

The following are multiple choice questions (with answers) about high school macroeconomics.

Classical economists believe
A. in the quantity theory of money—that both the velocity and the quantity of goods and services sold per period are fairly stable
B. there is a serious risk of a liquidity trap because the demand curve for money is relatively flat
C. that the government should make every effort to fine-tune the economy
D. that the aggregate supply curve is L-shaped
Answer:


## 6. Evaluation

In [16]:
backend = AIkenGPTBackend(
    model,
    tokenizer,
    model_identifier=MODEL_ID,
    tokenizer_identifier=TOKENIZER,
    scoring_method=SCORING_METHOD,
    text_reduction=TEXT_REDUCTION,
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_context_length=MAX_CONTEXT_LENGTH,
    checkpoint_sha256=checkpoint_hash,
)

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
results = evaluator.run(
    backend,
    OUTPUT_PATH,
    manifest_path=active_manifest_path,
)


1/10 high_school_macroeconomics[237]
2/10 high_school_government_and_politics[143]
3/10 high_school_chemistry[72]
4/10 sociology[154]
5/10 high_school_government_and_politics[140]
6/10 security_studies[62]
7/10 moral_disputes[95]
8/10 miscellaneous[455]
9/10 miscellaneous[87]
10/10 high_school_biology[58]


## 7. Results

In [17]:
from mmlu_eval.core import summarize

print(summarize(results))
print("Result CSV:", OUTPUT_PATH)
print("Manifest:", OUTPUT_PATH.with_suffix(".manifest.csv"))
print("Prompts:", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
display(results.head())

# OpenAI runとの入力一致を確認する場合:
# from mmlu_eval.compare import compare
# compare("/path/to/openai.prompts.jsonl", OUTPUT_PATH.with_suffix(".prompts.jsonl"))


{'n': 10, 'baseline_accuracy': 0.6, 'baseline_recall_A': 0.5, 'baseline_recall_B': 1.0, 'baseline_recall_C': 0.0, 'baseline_recall_D': 0.75, 'baseline_rstd': 0.369754986443726, 'debiased_accuracy': 0.4, 'debiased_recall_A': 0.25, 'debiased_recall_B': 1.0, 'debiased_recall_C': 0.0, 'debiased_recall_D': 0.5, 'debiased_rstd': 0.369754986443726, 'difference': -0.19999999999999996, 'position_A_prob': 0.27362571230805954, 'position_B_prob': 0.2409776721352265, 'position_C_prob': 0.23670372498650893, 'position_D_prob': 0.2486928905702049, 'wrong_to_correct': 0, 'correct_to_wrong': 2}
Result CSV: /content/drive/MyDrive/aikengpt_results/aikengpt_0shot_text_mean.csv
Manifest: /content/drive/MyDrive/aikengpt_results/aikengpt_0shot_text_mean.manifest.csv
Prompts: /content/drive/MyDrive/aikengpt_results/aikengpt_0shot_text_mean.prompts.jsonl


,subject,test_index,question,A,B,C,D,label,effective_ntrain,model_identifier,...,perm3_missing_count,perm3_error_bound,perm3_fifth_logprob,input_tokens,output_tokens,mean_error_bound,max_error_bound,has_all_missing,sample_order,run_id
0,high_school_macroeconomics,237,Classical economists believe,in the quantity theory of money—that both the ...,there is a serious risk of a liquidity trap be...,that the government should make every effort t...,that the aggregate supply curve is L-shaped,A,0,HayatoHongo/AIkenSGTv1:model_clean.safetensors,...,None,None,None,1908,0,None,None,False,0,06e00a8fccc63a07374ad7aaaeceb1303d0f29bfc5ab97...
1,high_school_government_and_politics,143,In which of the following cases did the Suprem...,Plessy v. Ferguson,McCulloch v. Maryland,Gibbons v. Ogden,Brown v. Board of Education,A,0,HayatoHongo/AIkenSGTv1:model_clean.safetensors,...,None,None,None,1436,0,None,None,False,1,06e00a8fccc63a07374ad7aaaeceb1303d0f29bfc5ab97...
2,high_school_chemistry,72,"A sample of oxygen gas at 50 °C is heated, rea...",Their velocity increases by a factor of two.,Their velocity increases by a factor of four.,Their kinetic energy increases by a factor of 2.,Their kinetic energy increases by a factor of ...,D,0,HayatoHongo/AIkenSGTv1:model_clean.safetensors,...,None,None,None,1824,0,None,None,False,2,06e00a8fccc63a07374ad7aaaeceb1303d0f29bfc5ab97...
3,sociology,154,The 'decentralized city' can be identified by:,the shift of employment and services away from...,the degendering of public space as women use l...,gentrification: the movement of middle class p...,all of the above,D,0,HayatoHongo/AIkenSGTv1:model_clean.safetensors,...,None,None,None,1708,0,None,None,False,3,06e00a8fccc63a07374ad7aaaeceb1303d0f29bfc5ab97...
4,high_school_government_and_politics,140,Which of the following is true of court cases ...,They are tried in civil court.,The federal court system has exclusive jurisdi...,They are tried in criminal court.,The state court system has exclusive jurisdict...,A,0,HayatoHongo/AIkenSGTv1:model_clean.safetensors,...,None,None,None,1512,0,None,None,False,4,06e00a8fccc63a07374ad7aaaeceb1303d0f29bfc5ab97...
